# 01 — Exploração inicial do dataset

**Projeto Sentinel** — detecção de anomalias em transações de cartão de crédito.

Objetivo deste notebook:
1. Carregar o dataset bruto e confirmar sua integridade.
2. Confirmar e quantificar o desbalanceamento de classes (a maior ameaça identificada na análise SWOT/GUT — ver `docs/SWOT_GUT.md`).
3. Explorar as distribuições de `Amount` e `Time`.
4. Registrar conclusões preliminares que vão orientar as decisões de pré-processamento e modelagem.

> Este notebook é só exploração — nenhum split de treino/teste ou balanceamento acontece aqui. Isso fica em `src/preprocessing.py`, usado pelos notebooks de modelagem.

In [ ]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir do notebook, que roda de dentro
# de notebooks/ (fora da raiz do projeto).
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import load_raw_data, class_distribution

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
df = load_raw_data()
df.shape

## 1. Visão geral do dataset

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Checagem de valores ausentes — o dataset original não tem, mas é a
# primeira coisa a verificar em qualquer EDA antes de confiar em estatísticas.
df.isna().sum().sum()

In [ ]:
df.describe()

## 2. Distribuição de classes

Aqui confirmamos o desbalanceamento citado na análise SWOT/GUT (~0,17% de fraudes). Isso é o dado mais importante deste notebook: qualquer decisão de modelagem daqui pra frente depende dele.

**Por que isso importa tanto:** um classificador que sempre prevê "transação legítima" já acerta ~99,8% das vezes (acurácia alta) e ainda assim é completamente inútil — nunca detecta uma fraude sequer. É por isso que o projeto usa matriz de confusão, precisão, recall, F1-score e AUC-ROC/AUC-PR como métricas principais, nunca acurácia isolada.

In [ ]:
dist = class_distribution(df)
dist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=df, x="Class", ax=axes[0])
axes[0].set_title("Contagem absoluta por classe")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Legítima", "Fraude"])

# Escala log no eixo Y — em escala linear a barra de fraudes é praticamente
# invisível ao lado da barra de transações legítimas.
sns.countplot(data=df, x="Class", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Mesma contagem, eixo Y em escala log")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Legítima", "Fraude"])

plt.tight_layout()
plt.show()

## 3. Estatísticas de `Amount`

In [ ]:
df.groupby("Class")["Amount"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df, x="Class", y="Amount", ax=axes[0])
axes[0].set_title("Amount por classe (escala original)")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Legítima", "Fraude"])

# Amount tem cauda longa (poucas transações de valor muito alto) — escala
# log ajuda a visualizar a distribuição sem esses outliers dominando o gráfico.
sns.boxplot(data=df, x="Class", y="Amount", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Amount por classe (escala log)")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Legítima", "Fraude"])

plt.tight_layout()
plt.show()

## 4. Estatísticas de `Time`

`Time` é o número de segundos decorridos desde a primeira transação do dataset (que cobre ~2 dias). Convertemos para horas para facilitar a leitura.

In [ ]:
df["Hour"] = (df["Time"] / 3600) % 24

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(
    data=df, x="Hour", hue="Class", bins=48, stat="density",
    common_norm=False, element="step", ax=ax,
)
ax.set_title("Distribuição de transações por hora do dia (normalizado por classe)")
ax.set_xlabel("Hora do dia (aproximada)")
plt.tight_layout()
plt.show()

## 5. Conclusões preliminares

_Preencher após rodar o notebook com o dataset real. Pontos a comentar:_

- Proporção exata de fraudes encontrada vs. o ~0,17% documentado.
- Diferenças relevantes na distribuição de `Amount` entre classes (ex.: fraudes concentradas em valores baixos?).
- Padrões por horário que possam virar feature (ex.: fraude mais comum em determinado período?).
- Implicações para a etapa de modelagem (`src/preprocessing.py` já aplica split estratificado + balanceamento só no treino, conforme decidido na análise de risco).